## Problem Statement

You are a data scientist at a startup. Over the past month, you ran 100 machine learning experiments trying different hyperparameters for a customer churn model. You logged everything to MLflow, but now your manager asks: "Which experiments had accuracy above 85% AND used less than 150 trees in the random forest?"

You have an MLflow tracking server running locally. Your experiments are stored in the default `mlruns` directory. Each experiment logged the following:

- Parameters: `n_estimators`, `max_depth`, `min_samples_split`
- Metrics: `accuracy`, `precision`, `recall`, `f1_score`

Write a Python script that:

1. Connects to your local MLflow tracking server
2. Retrieves all runs from the experiment named "customer_churn_prediction"
3. Filters runs where `accuracy > 0.85` AND `n_estimators < 150`
4. Prints the run IDs, accuracy values, and n_estimators for matching runs
5. Calculates and prints the average accuracy of these filtered runs

In [2]:
# sample experiment
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

# --- Generate dummy customer churn dataset ---
X, y = make_classification(
    n_samples=5000,
    n_features=20,
    n_informative=10,
    n_redundant=5,
    n_classes=2,
    flip_y=0.03,    # inject noise
    random_state=42
)

df = pd.DataFrame(X, columns=[f"feature_{i}" for i in range(20)])
df["churn"] = y

df.head()


,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,...,feature_11,feature_12,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19,churn
0,-0.305732,-3.951980,0.514882,-0.523283,0.630575,0.983941,-3.136115,-2.537985,4.716418,-0.761247,...,14.672024,-4.127115,7.516773,-1.144473,3.236354,-1.652023,-1.272756,-3.673646,-1.973476,0
1,0.727106,-0.890934,-4.194994,0.575102,1.952518,-0.843613,5.498002,0.454684,3.189835,1.237112,...,-4.033769,0.705786,-4.959869,7.451579,-2.150975,0.643432,2.208227,1.022050,2.768915,1
2,1.082043,-0.178114,-1.217761,-0.133503,0.210660,-0.284677,4.298803,-2.598830,4.340476,1.915736,...,3.942273,0.591490,-3.639847,3.539991,-1.078800,1.205765,5.166787,-0.908239,-0.407665,1
3,-1.160575,-0.078394,2.056583,0.456982,-0.265349,0.276672,-2.175385,2.646391,-0.513143,-0.713052,...,-3.078818,0.562132,-0.103900,0.832820,-1.677183,3.726429,-0.095043,0.319511,-0.772537,0
4,-0.369784,4.612814,-0.473455,1.522818,-0.051438,-0.077535,0.968228,-0.276592,-3.081819,0.895249,...,-7.642565,1.588959,-6.672679,0.888466,-0.093355,-3.042082,2.603982,1.042878,3.051317,0


In [ ]:
#log in mlflow
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import random

mlflow.set_experiment("churn_random_forest_sweep")

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    df.drop("churn", axis=1), df["churn"],
    test_size=0.2, random_state=42
)

for exp_id in range(100):
    n_trees = random.randint(50, 300)
    max_depth = random.choice([5, 10, 20, None])

    with mlflow.start_run(run_name=f"rf_exp_{exp_id}"):

        model = RandomForestClassifier(
            n_estimators=n_trees,
            max_depth=max_depth,
            random_state=exp_id
        )
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        acc = accuracy_score(y_test, preds)

        # Log hyperparameters and metrics
        mlflow.log_param("n_estimators", n_trees)
        mlflow.log_param("max_depth", max_depth)
        mlflow.log_metric("accuracy", acc)

        # Log model
        mlflow.sklearn.log_model(model, "model")

        print(f"Run {exp_id:03d} | Trees={n_trees} | Acc={acc:.4f}")


* Experiments are stored in ML Runs directory

In [4]:
# Task to do
experiment_name = "churn_random_forest_sweep"
experiment = mlflow.get_experiment_by_name(experiment_name) 

In [ ]:
# Retrieve all runs sorted by accuracy in descending order
all_runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.accuracy DESC"],
    max_results=1000,
    run_view_type=mlflow.entities.ViewType.ALL,
    output_format='pandas'
)


In [7]:
filtered_runs = all_runs[
    (all_runs['metrics.accuracy'] > 0.94) &
    (all_runs['params.n_estimators'].astype(int) < 150)
]
print(f"Found {len(filtered_runs)} runs matching the criteria.")


Found 5 runs matching the criteria.


In [8]:
for idx, row in filtered_runs.iterrows():
    print(f"Run ID: {row['run_id']}, Accuracy: {row['metrics.accuracy']}, N_Estimators: {row['params.n_estimators']}")

Run ID: 68a58f168bec49a6a6e5a263b953d22f, Accuracy: 0.944, N_Estimators: 148
Run ID: 39f2a07f39a64f27b1f923d7a8ab14e3, Accuracy: 0.942, N_Estimators: 121
Run ID: 23b3040bb6174326bb873f3d898a1704, Accuracy: 0.942, N_Estimators: 122
Run ID: ed44a5779e0947b9be62867e244df0c0, Accuracy: 0.941, N_Estimators: 74
Run ID: 457aad941f4642a4ab10fbc0d2d8e430, Accuracy: 0.941, N_Estimators: 115
